In [2]:
import pandas as pd

In [8]:
event_log = pd.read_parquet("~/jax_lob_sim/data/parquet/event_log/PFE_2504.parquet")
book_state = pd.read_parquet("~/jax_lob_sim/data/parquet/book_state/PFE_2504.parquet")
event_log.head()

,ts,level,size,action
0,1743516000018420653,-1,700,T
1,1743516000018471125,-1,100,A
2,1743516000018484476,-10,125,A
3,1743516000018572050,2,100,C
4,1743516000018682141,-1,100,A


In [9]:
book_state.head()

,ts,spread,imbalance,best_size,q-4,q-3,q-2,q-1,q+1,q+2,q+3,q+4,best_bid_px,best_ask_px
0,1743516000018317109,1,0.0704,1506,500,1210,1000,806,700,525,600,800,24.82,24.83
1,1743516000018420653,2,0.2111,1331,500,1210,1000,806,525,600,800,600,24.82,24.84
2,1743516000018471125,2,0.2662,1431,500,1210,1000,906,525,600,800,600,24.82,24.84
3,1743516000018484476,2,0.2662,1431,500,1210,1000,906,525,600,800,600,24.82,24.84
4,1743516000018572050,2,0.2662,1431,500,1210,1000,906,525,500,800,600,24.82,24.84


In [10]:
import numpy as np

def get_imbalance_bin(series):
    v = series.to_numpy()
    
    # Divide by 0.1 and round to fix floating-point precision issues
    # e.g., -0.1 / 0.1 could be -0.9999... instead of -1.0
    scaled = np.round(v / 0.1, 8)

    result = np.where(
        v == 0,                          # bin 0: exactly zero
        0,
        np.where(
            v < 0,
            np.floor(scaled).astype(int),  # negative: [0.1*i, 0.1*(i+1))
            np.ceil(scaled).astype(int)    # positive: (0.1*(i-1), 0.1*i]
        )
    )
    return result

book_state['imbalance_bin'] = get_imbalance_bin(book_state['imbalance'])

In [11]:
book_state.head()

,ts,spread,imbalance,best_size,q-4,q-3,q-2,q-1,q+1,q+2,q+3,q+4,best_bid_px,best_ask_px,imbalance_bin
0,1743516000018317109,1,0.0704,1506,500,1210,1000,806,700,525,600,800,24.82,24.83,1
1,1743516000018420653,2,0.2111,1331,500,1210,1000,806,525,600,800,600,24.82,24.84,3
2,1743516000018471125,2,0.2662,1431,500,1210,1000,906,525,600,800,600,24.82,24.84,3
3,1743516000018484476,2,0.2662,1431,500,1210,1000,906,525,600,800,600,24.82,24.84,3
4,1743516000018572050,2,0.2662,1431,500,1210,1000,906,525,500,800,600,24.82,24.84,3


In [13]:
book_state["imbalance_bin"].value_counts()

imbalance_bin
-4     6914
-3     6757
-2     6402
 1     6258
-5     6250
-1     6216
 2     6037
 3     5659
-6     5513
 4     4923
-7     4176
-8     4145
 5     3999
 6     3584
-9     3479
-10    3301
 7     2784
 8     1925
 9     1686
 10    1386
 0      118
Name: count, dtype: int64